In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#

# Pipelines in scikit-learn

### Author: Laura Krajčovičová (2026)

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

import sklearn
print("scikit-learn version:", sklearn.__version__)

# 1. Why pipelines

A **pipeline chains preprocessing steps and a final estimator into ONE object**. It behaves like a single model:
`fit`, `predict`, `predict_proba`, `score`, ...

Main benefits:
- **No data leakage** - during cross-validation every step (scaler, imputer, selector...) is re-fitted *only on the training fold*.
- **Cleaner code** - one `fit` / `predict` call instead of manually transforming train and test data.
- **Reproducibility** - the whole workflow (preprocessing + model) can be saved and deployed as one file.
- **Tuning** - preprocessing parameters can be tuned together with model parameters (`GridSearchCV`).

### Data leakage 
Leakage = information from the test data influences the training process (e.g. fitting the scaler on the *whole* dataset before splitting).
The scores then look better than they really are.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print(X.shape, np.bincount(y))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

**WRONG (leaky) way:** fit the scaler on everything, split afterwards / fit on train and test separately.

In [ ]:
# WRONG: the scaler "sees" the test data (mean/std computed on all rows)
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X)          # <- LEAKAGE!
Xtr_leak, Xte_leak, ytr_leak, yte_leak = train_test_split(
    X_all_scaled, y, test_size=0.25, stratify=y, random_state=42)

clf = LogisticRegression(max_iter=5000).fit(Xtr_leak, ytr_leak)
print("Leaky accuracy:", clf.score(Xte_leak, yte_leak))

**CORRECT way (manual):** `fit` / `fit_transform` ONLY on training data, `transform` on test data.
It works, but it is easy to make a mistake once there are many steps - that is where `Pipeline` helps.

In [ ]:
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(X_train)   # fit + transform on train
Xte_s = scaler.transform(X_test)        # ONLY transform on test

clf = LogisticRegression(max_iter=5000).fit(Xtr_s, y_train)
print("Manual (correct) accuracy:", clf.score(Xte_s, y_test))

# 2. Building a pipeline

Rules:
- All steps **except the last** must be *transformers* (have `fit` + `transform`).
- The **last** step can be anything - usually an estimator (classifier / regressor), but it can be a transformer too.
- Steps are a list of `(name, object)` tuples. Names must be unique and must **not contain double underscore `__`** (that has a special meaning, see section 5).

## 2.1 `Pipeline`

In [ ]:
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=0)),
])

pipe.fit(X_train, y_train)              # scaler.fit_transform -> clf.fit
y_pred = pipe.predict(X_test)           # scaler.transform -> clf.predict

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 2.2 `make_pipeline` 

Names are the lowercase class names: `standardscaler`, `logisticregression`. Shorter to write, but the names are less readable - for `GridSearchCV` prefer explicit names with `Pipeline`.

In [ ]:
pipe2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=0))
pipe2.fit(X_train, y_train)
print("Accuracy:", pipe2.score(X_test, y_test))
print("Step names:", list(pipe2.named_steps))

**What happens inside `fit` / `predict`?**

| Call | What the pipeline does |
|---|---|
| `pipe.fit(X, y)` | for each transformer: `fit_transform(X, y)`; the last step gets `fit(X_transformed, y)` |
| `pipe.predict(X)` | for each transformer: `transform(X)`; the last step gets `predict(X_transformed)` |
| `pipe.score(X, y)` | same as `predict`, then the final estimator's `score` |

A pipeline exposes only the methods that its **last step** has (e.g. `predict_proba` exists only if the final estimator has it).

# 3. Mixed data types: `ColumnTransformer`

Real datasets have numeric **and** categorical columns, missing values, ... Each column group needs different preprocessing.
`ColumnTransformer` applies different transformers to different columns and concatenates the results.

Typical structure:

```
Pipeline
 |- preprocessor  (ColumnTransformer)
 |    |- num: Pipeline(imputer -> scaler)       -> numeric columns
 |    |- cat: Pipeline(imputer -> one-hot)      -> categorical columns
 |- clf           (LogisticRegression)
```

In [ ]:
rng = np.random.default_rng(0)
n = 600

df = pd.DataFrame({
    "age":    rng.normal(40, 12, n).round(),
    "income": rng.lognormal(10, 0.5, n).round(),
    "city":   rng.choice(["Bratislava", "Kosice", "Zilina", "Nitra"], n),
    "plan":   rng.choice(["basic", "premium"], n, p=[0.7, 0.3]),
})
# target depends on the features (+ noise)
logit = 0.04*(df["age"]-40) + 0.00003*(df["income"]-25000) + 1.2*(df["plan"] == "premium") - 0.5
df["churn"] = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)

# inject missing values
df.loc[rng.choice(n, 40, replace=False), "age"] = np.nan
df.loc[rng.choice(n, 40, replace=False), "city"] = np.nan

df.head()

In [ ]:
df.info()
print(df.isna().sum())

In [ ]:
X = df.drop(columns="churn")
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

num_cols = ["age", "income"]
cat_cols = ["city", "plan"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # handle_unknown="ignore" - unseen categories in test data don't crash predict()
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",          # what to do with the other columns - "drop" (default) or "passthrough"
)

full_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(max_iter=1000)),
])

full_pipe.fit(X_train, y_train)
print("Accuracy:", full_pipe.score(X_test, y_test))

### Looking at what the preprocessor produced
`get_feature_names_out()` tells you the names of the output columns (after one-hot encoding etc.) - useful for inspecting model coefficients.

In [ ]:
feature_names = full_pipe.named_steps["preprocessor"].get_feature_names_out()
coefs = full_pipe.named_steps["clf"].coef_[0]

pd.Series(coefs, index=feature_names).sort_values().plot.barh(figsize=(7, 4))
plt.title("Logistic regression coefficients")
plt.tight_layout()
plt.show()

### Selecting columns automatically
Instead of hard-coding column names you can select by dtype:

In [ ]:
preprocessor_auto = ColumnTransformer([
    ("num", num_pipe, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipe, make_column_selector(dtype_exclude=np.number)),
])
auto_pipe = make_pipeline(preprocessor_auto, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print("Accuracy:", auto_pipe.score(X_test, y_test))

In [ ]:
# Output as a pandas DataFrame (sklearn >= 1.2) - handy for debugging
# (needs sparse_output=False in OneHotEncoder, see above)
full_pipe.named_steps["preprocessor"].set_output(transform="pandas")
full_pipe.named_steps["preprocessor"].fit_transform(X_train).head()

**Notes**
- The `ColumnTransformer` expects the **same column names** (when using a DataFrame) at `predict` time as at `fit` time.
- `remainder="passthrough"` keeps the columns you did not list, otherwise they are dropped.

# 4. Cross-validation and `GridSearchCV` with pipelines

## 4.1 Cross-validation

Passing the **whole pipeline** to `cross_val_score` guarantees that imputer, scaler, encoder... are fitted only on the training part of each fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(full_pipe, X_train, y_train, cv=cv, scoring="roc_auc")
print("ROC-AUC per fold:", scores.round(3))
print(f"Mean: {scores.mean():.3f} +/- {scores.std():.3f}")

## 4.2 Hyperparameter tuning

Parameter names in the grid follow the pattern **`<step name>__<parameter name>`**, with as many `__` levels as the nesting requires:

- `clf__C` - parameter `C` of the step called `clf`
- `preprocessor__num__imputer__strategy` - parameter `strategy` of `imputer` inside `num` inside `preprocessor`

You can also put **whole objects** into the grid (e.g. try different scalers, or `"passthrough"`).

In [ ]:
param_grid = {
    "preprocessor__num__imputer__strategy": ["mean", "median"],
    "preprocessor__num__scaler": [StandardScaler(), "passthrough"],
    "clf__C": [0.01, 0.1, 1, 10],
}

grid = GridSearchCV(
    full_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", round(grid.best_score_, 3))
print("Test score (best model):", round(grid.score(X_test, y_test), 3))

In [ ]:
# Results as a table
res = pd.DataFrame(grid.cv_results_)
res[["params", "mean_test_score", "std_test_score", "rank_test_score"]] \
    .sort_values("rank_test_score").head(5)

In [ ]:
# grid.best_estimator_ is the full pipeline re-fitted on the whole training set (refit=True default)
best_pipe = grid.best_estimator_
best_pipe.predict(X_test)[:10]

**Notes**
- You may see a `ConvergenceWarning` during this grid: it comes from the `"passthrough"` option (unscaled `income` has huge values, so the solver converges badly). This is exactly why scaling matters for logistic regression.
- Use the **test set only once**, at the very end. Tune with cross-validation on the training data.
- If you get `Invalid parameter ... for estimator Pipeline`, check the names with `pipe.get_params().keys()`.
- Large grids get expensive quickly: try `RandomizedSearchCV` (same API, `n_iter=...`) or `HalvingGridSearchCV`.
- `Pipeline(..., memory="cache_dir")` caches fitted transformers - useful when only the final estimator's parameters change and preprocessing is slow.
- Use `scoring=` that fits the problem (e.g. `f1`, `roc_auc`, `average_precision` for imbalanced data) - accuracy can be misleading.

# 5. Feature selection inside a pipeline

Feature selection is a common source of leakage: if you select the "best" features using the **whole** dataset and cross-validate afterwards, the CV score is optimistically biased. Put the selector into the pipeline!

Back to the breast cancer dataset (30 numeric features).

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# Filter method: SelectKBest (univariate statistical test)
pipe_filter = Pipeline([
    ("scaler", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif, k=10)),
    ("clf", LogisticRegression(max_iter=5000)),
])

# Embedded method: L1-penalised linear model chooses the features
pipe_embedded = Pipeline([
    ("scaler", StandardScaler()),
    ("select", SelectFromModel(LinearSVC(C=0.05, penalty="l1", dual=False, max_iter=10000))),
    ("clf", LogisticRegression(max_iter=5000)),
])

pipe_all = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000)),
])

for name, p in [("all features", pipe_all), ("SelectKBest(10)", pipe_filter), ("SelectFromModel(L1)", pipe_embedded)]:
    s = cross_val_score(p, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name:22s} ROC-AUC = {s.mean():.3f} +/- {s.std():.3f}")

In [ ]:
# Which features were selected? Use the fitted selector (support mask + feature names)
pipe_filter.fit(X_train, y_train)
mask = pipe_filter.named_steps["select"].get_support()
print("Selected features:")
print(np.array(data.feature_names)[mask])

In [ ]:
# Tune k together with C
grid = GridSearchCV(
    pipe_filter,
    {"select__k": [5, 10, 15, 20, 30], "clf__C": [0.1, 1, 10]},
    cv=5, scoring="roc_auc", n_jobs=-1,
).fit(X_train, y_train)
print(grid.best_params_, round(grid.best_score_, 3))

**Notes**
- Order matters: scale -> select -> model. (Scaling before L1-based selection is important, otherwise the penalty is not comparable across features.)
- All steps with `get_support()` can be inspected after fitting (`SelectKBest`, `SelectFromModel`, `RFE`, ...).

**Note:** Never compute statistics (mean, quantiles, category lists...) in `transform` - they must be computed in `fit`, otherwise you get leakage and inconsistent results between train and test.

# 6. Saving a pipeline

Because the whole workflow is one object, you save/load **one file** and get preprocessing + model together.

In [ ]:
import joblib

best = grid.best_estimator_
joblib.dump(best, "model_pipeline.joblib")

loaded = joblib.load("model_pipeline.joblib")
print("Same predictions after loading:", np.array_equal(best.predict(X_test), loaded.predict(X_test)))